# Test thủ công từng vị trí tay gắp + bbox đơn giản

File này là bản tách cell rõ ràng để test thủ công.

- Cell 1: khai báo tuple góc.
- Cell 2: setup hàm `send_arm(...)`.
- Cell 3-10: mỗi cell test đúng 1 pose.
- Cell 12: chụp 1 ảnh camera.
- Cell 14/15: nhập hoặc kéo bbox.
- Cell 16: vẽ bbox.
- Cell 17: lưu JSON.

Muốn chạy servo thật: đổi `RUN_ARM_HARDWARE = True` ở Cell 1.

In [ ]:
# CELL 1 - Khai báo góc và cấu hình chung

from pathlib import Path
import json
import time
from datetime import datetime

import cv2
import matplotlib.pyplot as plt

# Góc servo theo thứ tự: K1 K2 K3 K4 K5 K6
DEFAULT_ANGLES = (90, 180, 0, 50, 90, 90)
PICK_ANGLES_1 = (0, 180, 0, 50, 90, 90)
PICK_ANGLES_2 = (90, 180, 0, 50, 90, 90)
PICKING_ANGLES_1 = (0, 90, 90, 60, 90, 30)
PICKING_ANGLES_2 = (90, 90, 90, 60, 90, 30)

GRASP_ANGLES_1 = (0, 90, 90, 60, 90, 132)
PICKED_ANGLES_1 = (0, 180, 0, 50, 90, 132)
HOLD_ANGLES = (90, 180, 0, 50, 90, 132)

PROJECT_DIR = Path('/home/pi/chay')
IMAGE_PATH = PROJECT_DIR / 'pick1_manual.jpg'
BBOX_IMAGE_PATH = PROJECT_DIR / 'pick1_manual_bbox.jpg'
BBOX_JSON_PATH = PROJECT_DIR / 'pick1_gripper_box_candidate.json'

CAMERA_ID = 0
COLOR = 'yellow'
RUN_TIME_MS = 10000
RUN_ARM_HARDWARE = True  # đổi True nếu muốn chạy servo thật

print('RUN_ARM_HARDWARE =', RUN_ARM_HARDWARE)

RUN_ARM_HARDWARE = True


In [13]:
# CELL 2 - Setup arm và hàm gửi góc

bot = None


def connect_arm():
    global bot
    if bot is None:
        from Rosmaster_Lib import Rosmaster
        bot = Rosmaster()
        bot.create_receive_threading()
        bot.set_uart_servo_torque(True)
        time.sleep(0.5)
        print('Đã kết nối Rosmaster arm')
    return bot


def send_arm(angles, run_time_ms=RUN_TIME_MS):
    angles = tuple(int(a) for a in angles)
    print('Send arm:', angles)

    if not RUN_ARM_HARDWARE:
        print('[DRY-RUN] Chưa chạy servo thật. Đổi RUN_ARM_HARDWARE = True ở CELL 1 nếu cần.')
        return

    arm = connect_arm()
    arm.set_uart_servo_angle_array(list(angles), run_time=run_time_ms)
    time.sleep(run_time_ms / 1000.0 + 0.5)
    print('Done')

## Test từng vị trí tay gắp

Bấm chạy từng cell dưới đây để test đúng một vị trí. Nếu muốn đổi góc thì sửa tuple ở CELL 1 trước.

In [6]:
# CELL 3 - Test DEFAULT_ANGLES
send_arm(DEFAULT_ANGLES)

Send arm: (90, 180, 0, 50, 90, 90)
Rosmaster Serial Opened! Baudrate=115200
----------------create receive threading--------------
Đã kết nối Rosmaster arm


KeyboardInterrupt: 

In [16]:
# CELL 4 - Test PICK_ANGLES_1
send_arm(PICK_ANGLES_1)

Send arm: (0, 180, 0, 50, 90, 90)


KeyboardInterrupt: 

In [17]:
# CELL 5 - Test PICKING_ANGLES_1
send_arm(PICKING_ANGLES_1)

Send arm: (0, 90, 90, 60, 90, 30)
Done


In [ ]:
# CELL 6 - Test GRASP_ANGLES_1
send_arm(GRASP_ANGLES_1)

In [ ]:
# CELL 7 - Test PICKED_ANGLES_1
send_arm(PICKED_ANGLES_1)

In [ ]:
# CELL 8 - Test HOLD_ANGLES
send_arm(HOLD_ANGLES)

In [ ]:
# CELL 9 - Test PICK_ANGLES_2 nếu cần
send_arm(PICK_ANGLES_2)

In [ ]:
# CELL 10 - Test PICKING_ANGLES_2 nếu cần
send_arm(PICKING_ANGLES_2)

## Chụp ảnh camera

Đưa tay gắp tới vị trí muốn test, đặt vật đúng chỗ, rồi chạy cell chụp ảnh.

In [ ]:
# CELL 11 - Hàm hiển thị ảnh

def show_image(path, title='image'):
    img = cv2.imread(str(path))
    if img is None:
        print('Không đọc được ảnh:', path)
        return
    plt.figure(figsize=(8, 6))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis('off')
    plt.show()

In [ ]:
# CELL 12 - Chụp 1 ảnh từ camera

cap = cv2.VideoCapture(CAMERA_ID)
if not cap.isOpened():
    raise RuntimeError(f'Không mở được camera {CAMERA_ID}')

frame = None
for _ in range(20):
    ok, frame = cap.read()
    if not ok:
        cap.release()
        raise RuntimeError('Không đọc được frame camera')
    time.sleep(0.03)

ok, frame = cap.read()
cap.release()
if not ok:
    raise RuntimeError('Không đọc được frame cuối')

cv2.imwrite(str(IMAGE_PATH), frame)
print('Đã lưu:', IMAGE_PATH)
show_image(IMAGE_PATH, 'captured')

In [ ]:
# CELL 13 - Xem lại ảnh đã chụp
show_image(IMAGE_PATH, 'captured image')

## Chọn và vẽ bbox

Có 2 cách:

- Cách nhanh: tự nhập `BBOX = (x, y, w, h)`.
- Cách kéo chuột: chạy cell `cv2.selectROI`, cần desktop/VNC có cửa sổ GUI.

In [ ]:
# CELL 14 - Cách 1: tự nhập bbox pixel
# Format: BBOX = (x, y, width, height)
# Bbox chuẩn lấy từ /home/pi/chay/pick1_manual.jpg tại vị trí gắp lý tưởng.

BBOX = (247, 209, 155, 127)
print('BBOX =', BBOX)

In [ ]:
# CELL 15 - Cách 2: kéo bbox bằng chuột trên ảnh
# Nhấn Enter/Space để xác nhận, nhấn c để hủy.

img = cv2.imread(str(IMAGE_PATH))
if img is None:
    raise RuntimeError(f'Không đọc được ảnh: {IMAGE_PATH}')

roi = cv2.selectROI('chon bbox roi Enter', img, showCrosshair=True, fromCenter=False)
cv2.destroyAllWindows()

BBOX = tuple(int(v) for v in roi)
print('BBOX =', BBOX)

In [ ]:
# CELL 16 - Vẽ bbox lên ảnh để kiểm tra

img = cv2.imread(str(IMAGE_PATH))
if img is None:
    raise RuntimeError(f'Không đọc được ảnh: {IMAGE_PATH}')

x, y, w, h = [int(v) for v in BBOX]
out = img.copy()
cv2.rectangle(out, (x, y), (x + w, y + h), (255, 0, 255), 3)
cv2.circle(out, (x + w // 2, y + h // 2), 6, (255, 0, 255), -1)
cv2.imwrite(str(BBOX_IMAGE_PATH), out)

print('Đã lưu:', BBOX_IMAGE_PATH)
show_image(BBOX_IMAGE_PATH, f'bbox={BBOX}')

In [ ]:
# CELL 17 - Lưu bbox thành JSON cho visual_approach_node.py

img = cv2.imread(str(IMAGE_PATH))
if img is None:
    raise RuntimeError(f'Không đọc được ảnh: {IMAGE_PATH}')

frame_h, frame_w = img.shape[:2]
x, y, w, h = [int(v) for v in BBOX]
area = float(w * h)

config = {
    'created_at': datetime.now().isoformat(timespec='seconds'),
    'camera_id': CAMERA_ID,
    'color': COLOR,
    'frame_size': {
        'width': frame_w,
        'height': frame_h,
    },
    'roi': [0.0, 0.0, 1.0, 0.8],
    'object_bbox_px': {
        'x': x,
        'y': y,
        'width': w,
        'height': h,
        'center_x': x + w // 2,
        'center_y': y + h // 2,
        'area': area,
    },
    'gripper_box': [
        round(x / frame_w, 6),
        round(y / frame_h, 6),
        round(w / frame_w, 6),
        round(h / frame_h, 6),
    ],
    'ready_min_area': round(area * 0.80, 2),
    'ready_min_height': max(1, int(h * 0.80)),
    'center_tolerance_ratio': 0.08,
    'source': 'manual_gripper_bbox_tune.ipynb',
    'debug_image': str(BBOX_IMAGE_PATH),
}

with BBOX_JSON_PATH.open('w', encoding='utf-8') as f:
    json.dump(config, f, indent=2)
    f.write('\n')

print('Đã lưu:', BBOX_JSON_PATH)
print(json.dumps(config, indent=2))

## Test file bbox vừa lưu

```bash
cd /home/pi/chay
python3 -B new/visual_approach_node.py --camera-id 0 --color yellow --roi 0 0 1 0.8 --gripper-config pick1_gripper_box_candidate.json --show --continuous
```

Khi candidate ổn:

```bash
cp /home/pi/chay/pick1_gripper_box_candidate.json /home/pi/chay/pick1_gripper_box.json
```